# RAG-Based Banking Document Assistant
**Author:** Shanmukh Bysani

A Retrieval-Augmented Generation (RAG) chatbot that answers natural-language questions over banking policy documents. It chunks documents, embeds them with Sentence Transformers, stores them in ChromaDB for semantic search, and uses an LLM to generate grounded answers **with source attribution**.

**How to run:** run the cell, paste your free Groq key when prompted (console.groq.com/keys). An interactive UI appears in the output.

**Tech:** LangChain - ChromaDB - Sentence Transformers - Groq (LLM) - Gradio

In [ ]:
# ============================================================================
#  RAG-BASED BANKING DOCUMENT ASSISTANT  --  all-in-one interactive app
#  Chunk (LangChain) -> embed (Sentence Transformers) -> search (ChromaDB)
#  -> grounded answer with sources (LLM)
# ============================================================================

!pip install -q groq gradio chromadb sentence-transformers langchain langchain-text-splitters

# ---- sqlite fix: ChromaDB needs a recent sqlite; swap in pysqlite3 if needed -
import sys
try:
    import chromadb
except Exception:
    !pip install -q pysqlite3-binary
    __import__('pysqlite3')
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')
    import chromadb

import os, getpass
from groq import Groq
from sentence_transformers import SentenceTransformer
import gradio as gr

# LangChain moved the splitter between versions -- handle both
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

# ---- API key -----------------------------------------------------------------
api_key = os.environ.get("GROQ_API_KEY") or getpass.getpass("Paste your Groq API key: ")
client = Groq(api_key=api_key)
MODEL = "llama-3.3-70b-versatile"   # check console.groq.com/docs/models if it errors

# ---- 1. Knowledge base: synthetic banking policy documents -------------------
DOCUMENTS = {
    "Accounts & Fees": """Our Everyday Current Account has no monthly maintenance fee.
The Premier Account costs GBP 10 per month and includes travel insurance and a higher overdraft limit.
There is no fee for withdrawing cash at any ATM within the UK. Withdrawals abroad incur a 2.75% foreign
transaction fee. Standard bank transfers within the UK are free via Faster Payments, usually arriving
within two hours.""",

    "Overdrafts": """An arranged overdraft lets you borrow up to an agreed limit on your current account.
The representative interest rate on arranged overdrafts is 39.9% EAR variable. There is a GBP 50
interest-free buffer on the Premier Account. If you go over your limit, we send a text alert and give you
until the end of the day to deposit funds before any charge applies.""",

    "Fraud & Security": """If you notice a transaction you do not recognise, contact us immediately so we
can freeze your card. Under the Contingent Reimbursement Model, customers who fall victim to authorised
push payment scams may be reimbursed if they took reasonable care. We will never call or text you asking
for your full PIN, password, or one-time passcode. Our fraud detection system monitors transactions in
real time and may temporarily block a payment that looks unusual until we verify it with you.""",

    "Loans & Mortgages": """Personal loans are available from GBP 1,000 to GBP 25,000 with terms from 1 to
7 years. The representative APR is 6.9% for loans between GBP 7,500 and GBP 15,000. Mortgage applications
require proof of income, a credit check, and a deposit of at least 5% of the property value. You can make
overpayments of up to 10% of your mortgage balance each year without an early repayment charge.""",

    "Cards": """Your debit card can be used contactless for payments up to GBP 100 per transaction.
Credit card customers benefit from up to 56 days interest-free on purchases if the balance is paid in
full. Lost or stolen cards can be frozen instantly in the mobile app. A replacement card arrives within
3 to 5 working days. We offer virtual cards for safer online shopping.""",

    "Complaints": """If you are unhappy with our service, you can raise a complaint by phone, in branch,
or through the mobile app. We aim to resolve complaints within three business days. If we cannot, we send
a written acknowledgement and aim to provide a final response within eight weeks. If you remain
dissatisfied, you can refer your complaint to the Financial Ombudsman Service free of charge.""",
}

# ---- 2. Chunk the documents (LangChain) --------------------------------------
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks, metadatas = [], []
for title, text in DOCUMENTS.items():
    for piece in splitter.split_text(text):
        chunks.append(piece)
        metadatas.append({"source": title})
print(f"Split {len(DOCUMENTS)} documents into {len(chunks)} chunks.")

# ---- 3. Embed chunks with Sentence Transformers ------------------------------
print("Loading embedding model (downloads ~80MB on first run)...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
chunk_embeddings = embedder.encode(chunks).tolist()

# ---- 4. Store in ChromaDB (vector database) ----------------------------------
chroma = chromadb.Client()
collection = chroma.get_or_create_collection("banking")
# clear any existing data, then add fresh
if collection.count() > 0:
    chroma.delete_collection("banking")
    collection = chroma.get_or_create_collection("banking")
collection.add(
    documents=chunks,
    embeddings=chunk_embeddings,
    metadatas=metadatas,
    ids=[f"chunk_{i}" for i in range(len(chunks))],
)
print(f"Indexed {collection.count()} chunks in ChromaDB.")

# ---- 5. Retrieval ------------------------------------------------------------
def retrieve(question, k=3):
    q_emb = embedder.encode([question]).tolist()
    res = collection.query(query_embeddings=q_emb, n_results=k)
    return res["documents"][0], [m["source"] for m in res["metadatas"][0]]

# ---- 6. Generation: answer using ONLY retrieved context ----------------------
def answer_question(question):
    if not question or not question.strip():
        return "*Type a question above and click Ask.*"
    try:
        docs, sources = retrieve(question)
        context = "\n\n".join(f"[{s}] {d}" for s, d in zip(sources, docs))
        prompt = f"""You are a helpful banking assistant. Answer the customer's question using ONLY the
context below. If the answer is not in the context, say you don't have that information and suggest they
contact the bank. Keep the answer concise.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
        )
        ans = resp.choices[0].message.content.strip()
        unique_sources = ", ".join(dict.fromkeys(sources))
        return f"{ans}\n\n---\n*Sources: {unique_sources}*"
    except Exception as e:
        return (f"### Something went wrong\n```\n{type(e).__name__}: {e}\n```\n"
                f"If this mentions the model, change MODEL to a current name from "
                f"console.groq.com/docs/models (e.g. `llama-3.1-8b-instant`).")

# ---- 7. Interactive UI (simple, version-safe pattern) ------------------------
EXAMPLES = [
    "What is the interest rate on an arranged overdraft?",
    "How long does a replacement card take?",
    "Can I be reimbursed if I'm scammed?",
    "What's the APR on a personal loan?",
    "How do I make a complaint?",
]

with gr.Blocks(title="Banking Document Assistant") as demo:
    gr.Markdown("# Banking Document Assistant (RAG)")
    gr.Markdown("Ask a question about banking policies. Answers are grounded in the document "
                "knowledge base and cite their sources.")
    with gr.Row():
        with gr.Column():
            question = gr.Textbox(label="Your question",
                                  placeholder="e.g. What is the overdraft interest rate?")
            ask = gr.Button("Ask", variant="primary")
            gr.Examples(EXAMPLES, inputs=question)
        with gr.Column():
            answer = gr.Markdown(value="*Ask a question to get a grounded answer with sources.*")
    ask.click(answer_question, inputs=question, outputs=answer)
    question.submit(answer_question, inputs=question, outputs=answer)

demo.queue()
demo.launch(share=True, debug=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indexed 12 chunks in ChromaDB.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://93a58090f29e8166a2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
